# Demo 19 — Intercept-only models: the mean, the one-sample *t*-test and the ICC

The simplest model has no predictor at all. Every value is one common number
plus a deviation:

```
Reaction ~ 1                      y = b0 + e
```

Its only coefficient, `b0`, is the mean, and testing it against a reference value
(`options.test_value`) is the **one-sample *t*-test**.

The data are the two baseline days of `sleepstudy` (lme4; Belenky et al. 2003):
18 subjects, reaction time measured twice each before the sleep restriction
started. The question: does the mean baseline reaction time differ from 250 ms?

## Run on Google Colab

On [Google Colab](https://colab.research.google.com)? Run the cell below first —
it installs kbstatpy, its R packages, and the demo data (~3–5 min the first time).
It is a no-op when you run this notebook locally from the kbstatpy source tree.
Then run the cells below to see the tables and figures rendered inline.

In [ ]:
# Google Colab only: install kbstatpy + its R packages + the demo data.
# (Does nothing when the notebook runs locally from the source tree.)
import sys
if 'google.colab' in sys.modules:
    !curl -sSL https://raw.githubusercontent.com/kimbostroem/kbstatpy/master/demos/colab_setup.sh | bash

## Setup

In [ ]:
import os

from kbstatpy import Kbstat, KbstatOptions

## The options

`x = ''` means no factor: kbstatpy fits `Reaction ~ 1`. `out_dir` is empty, so
results render inline only.

In [ ]:
def options(id_var):
    o = KbstatOptions()
    o.in_file     = os.path.join(o.demo_dir, 'data/sleepstudy.csv')
    o.out_dir     = ''            # inline only; set a folder to also save
    o.constraints = 'Days <= 1'   # the two baseline days only
    o.y           = 'Reaction'
    o.y_units     = 'ms'
    o.x           = ''            # no factor: the intercept-only model
    o.id          = id_var        # '' = no random effect
    o.test_value  = 250           # test the mean against 250 ms
    return o

## 1. Every row counted as independent

`Reaction ~ 1` with 36 rows: the one-sample *t*-test with df = 35. The plot is a
single violin with the mean and its CI; the dashed line is the test value.

In [ ]:
one_sample = Kbstat(options(''))
one_sample.run();

## 2. A random intercept per subject

The 36 rows are 18 persons measured twice, and two values of the same person are
more alike than two values of different persons. A random intercept says so:

```
Reaction ~ 1 + (1 | Subject)      y = b0 + u_subject + e
```

The mean stays the same, but the test now counts 18 subjects, not 36 rows.

In [ ]:
random_intercept = Kbstat(options('Subject'))
random_intercept.run();

## Interpretation

- **Same mean, different evidence.** Both models estimate the same mean (about
  261 ms). Ignoring the subjects gives *t*(35) and p ≈ 0.06; counting the
  subjects gives *t*(17) and p ≈ 0.16, with a wider CI. The first model's df were
  inflated by **pseudo-replication**: the second measurement of a person is not
  a new, independent person.
- **The ICC.** The second model splits the variance into the part between
  subjects and the part within them. `VARIANCE COMPONENTS` in the summary reports
  their ratio, the intraclass correlation (about 0.72): most of the variation in
  baseline reaction time is between persons, so a single measurement ranks
  persons reliably. With persons as units and repeated measurements of the same
  quantity, this is the test-retest reliability, ICC(1).
- **Cohen's d** divides the distance from 250 ms by the total SD of y (subject
  variance plus residual), so it is about the same in both models: the random
  intercept changes the precision, not the size of the effect.
- **`test_value`** is in the units of y. Without it the mean is tested against
  0, which is what a column of differences is tested against.